In [45]:
#Dataframe für die Bestimmung von m erstellen

import re
import pandas as pd
from pathlib import Path

#  Datei einlesen
txt_path = Path("Ergebnis aus Bestimmung von m")
lines = txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()

# DataFrame parsen
rows = []

dataset = None
p_val = None
m_from = None
m_to = None

for line in lines:
    line = line.strip()
    if line == "":
        continue

    # 1) "Iris Datensatz"
    m = re.match(r"^(.*)\s+Datensatz$", line)
    if m:
        dataset = m.group(1).strip()
        continue

    # 2) "p =  1.3"
    m = re.match(r"^p\s*=\s*([0-9.]+)$", line)
    if m:
        p_val = float(m.group(1))
        continue

    # 3) "Vergleich m=5 und m=10"
    m = re.match(r"^Vergleich\s+m\s*=\s*(\d+)\s+und\s+m\s*=\s*(\d+)\s*$", line)
    if m:
        m_from = int(m.group(1))
        m_to = int(m.group(2))
        continue

    # 4) Danach kommt eine Zahl (Höhe der Änderung)
    if re.match(r"^[+-]?\d+(\.\d+)?([eE][+-]?\d+)?$", line):
        height = float(line)

        rows.append({
            "dataset": dataset,
            "p": p_val,
            "m_from": m_from,
            "m_to": m_to,
            "change": f"{m_from}→{m_to}",
            "height": height
        })

df = pd.DataFrame(rows)

# Sortieren
df = df.sort_values(["dataset", "p", "m_from", "m_to"]).reset_index(drop=True)

# CSV speichern
out_csv = Path("Bestimmung_m_changes.csv")
df.to_csv(out_csv, index=False)

print("Fertig! DataFrame gespeichert als:", out_csv)
print(df.head())

Fertig! DataFrame gespeichert als: Bestimmung_m_changes.csv
     dataset    p  m_from  m_to    change     height
0  Covertype  1.0       5    10      5→10  14.204677
1  Covertype  1.0      10    50     10→50   4.842969
2  Covertype  1.0      50   100    50→100   1.595472
3  Covertype  1.0     200   500   200→500   1.371639
4  Covertype  1.0     500  1000  500→1000   1.168732


In [73]:
#Plots für die Bestimmung von m

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1) Alles auf Standard zurücksetzen (wichtig, wenn vorher ein Dark-Style aktiv war)
plt.style.use("default")

# Schriftgrößen anpassen
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 12

# DataFrame einlesen
df = pd.read_csv("Bestimmung_m_changes.csv")

# Plots erstellen
out_dir = Path("Plots")
out_dir.mkdir(exist_ok=True)

for dataset_name in df["dataset"].unique():
    sub = df[df["dataset"] == dataset_name]

    # x-Reihenfolge festlegen (z.B. 5→10, 10→50, 50→100, ...)
    order = (sub[["m_from", "m_to", "change"]]
             .drop_duplicates()
             .sort_values(["m_from", "m_to"]))
    x_labels = order["change"].tolist()

    # 2 Panels nebeneinander
    fig, (ax_lin, ax_log) = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

    # Für jedes p eine Linie
    for p_val in sorted(sub["p"].unique()):
        s2 = sub[sub["p"] == p_val].set_index("change").reindex(x_labels)
        y = s2["height"].values

        ax_lin.plot(x_labels, y, marker="o", label=f"p={p_val:g}")
        ax_log.plot(x_labels, y, marker="o", label=f"p={p_val:g}")

    # Linkes Panel: linear
    ax_lin.set_title("Linear")
    ax_lin.grid(True, linestyle="--", alpha=0.6)

    # Rechtes Panel: log
    ax_log.set_title("Log")
    ax_log.set_yscale("log")
    ax_log.grid(True, which="both", linestyle="--", alpha=0.6)

    # gemeinsame Legende (oben)
    handles, labels = ax_lin.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=min(5, len(labels)), bbox_to_anchor=(0.5, 0.98))

    # x labels besser lesbar
    for ax in (ax_lin, ax_log):
        ax.tick_params(axis="x", rotation=30)

    fig.suptitle(f"{dataset_name}", y=1.04, fontsize=20)
    fig.supxlabel("Änderung von m_1 zu m_2", fontsize=16)
    fig.supylabel("Höhe der Änderungen", fontsize=16)
    fig.tight_layout()

    out_dir = Path("Plots") / "Plots_Bestimmung_m_zwei_Panels"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{dataset_name}_two_panels.png"
    fig.savefig(out_path, dpi=200, bbox_inches="tight", transparent=False)
    plt.close(fig)

    print("Gespeichert:", out_path)

Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels\Covertype_two_panels.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels\Example2D_two_panels.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels\Iris_two_panels.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels\KDDCup_two_panels.png
Gespeichert: Plots\Plots_Bestimmung_m_zwei_Panels\Webspam_two_panels.png
